In [4]:
import pandas as pd

df_ext = pd.read_csv("Labdataset.csv", low_memory=False)

print("Shape:", df_ext.shape)
print("\nColumns:")
for c in df_ext.columns.tolist():
    print(f"  - {c}")

print("\nFirst 3 rows:")
df_ext.head(3)

Shape: (1048575, 16)

Columns:
  - Time
  - Source
  - Destination Address
  - Target Address
  - Protocol
  - Length
  - Type
  - Code
  - Length_1
  - Hop Limit
  - Payload Length
  - Next Header
  - Frame Number
  - Info
  - Class
  - Unnamed: 15

First 3 rows:


,Time,Source,Destination Address,Target Address,Protocol,Length,Type,Code,Length_1,Hop Limit,Payload Length,Next Header,Frame Number,Info,Class,Unnamed: 15
0,0.000000,Cisco_b8:ef:90,NaN,NaN,LOOP,60,NaN,NaN,NaN,NaN,NaN,NaN,1,Reply,Normal,0
1,1.864592,169.254.220.75,NaN,NaN,SSDP,217,NaN,NaN,NaN,NaN,NaN,NaN,2,M-SEARCH * HTTP/1.1,Normal,0
2,2.880087,169.254.220.75,NaN,NaN,SSDP,217,NaN,NaN,NaN,NaN,NaN,NaN,3,M-SEARCH * HTTP/1.1,Normal,0


In [3]:
for col in df_ext.columns:
    name = col.lower()
    if any(k in name for k in ["label", "class", "attack", "category", "type"]):
        print(f"=== {col} ===")
        print(df_ext[col].value_counts())
        print()

=== Type ===
Type
Echo (ping) reply          523016
Echo (ping) request        523015
Destination Unreachable       287
Neighbor Solicitation         156
Neighbor Advertisement        156
Router Advertisement           41
Name: count, dtype: int64

=== Class ===
Class
Attack    1045624
Normal       2951
Name: count, dtype: int64



In [7]:
!python ids_pipeline_v18.py --mode train --pcap /ipv6_research/raw_capture.pcap

MODE: TRAIN
  PCAP            : /ipv6_research/raw_capture.pcap
  Model bundle out: ./ids_model_v18.joblib

──────────────────────────────────────────────────────────────────────
STEP 1 — Parsing in-band markers from PCAP
──────────────────────────────────────────────────────────────────────
  Marker packets found: 0
  Phase intervals     : 0

  [WARN] No in-band markers found — PCAP was captured with the
         v4 bash script. Trying v4 CSV fallback ...

  events.csv path : \ipv6_research\raw_capture_events.csv
  nodes.csv  path : \ipv6_research\raw_capture_nodes.csv
  Loaded nodes.csv  → router=True, attacker=True, victims=20
  Loaded events.csv → 57 rows → 27 phase intervals

  v4 CSV fallback succeeded — 27 phase intervals loaded.

  Attacker MAC : aa:c1:ab:68:a5:ec
  Router   MAC : aa:c1:ab:36:e8:c9
  Victims      : 20

  Phase timeline:
    1776113555.9 → 1776113575.9  [Normal          ]  (warmup)
    1776113576.0 → 1776113926.0  [Normal          ]  (baseline_1)
    1776113928.

# Verify

In [8]:
import os
print("Bundle exists:", os.path.exists("./ids_model_v18.joblib"))
print("Size (MB):", round(os.path.getsize("./ids_model_v18.joblib") / 1024 / 1024, 2))

Bundle exists: True
Size (MB): 12.58


In [5]:
import sys, os, joblib

# Make the pipeline script importable (adjust path if needed)
sys.path.insert(0, ".")          # or the folder where ids_pipeline_v18.py lives

# Import the module — this defines _SafeXGB inside ids_pipeline_v18
import ids_pipeline_v18

# Register _SafeXGB in __main__ so joblib/pickle can find it during loading
import __main__
__main__._SafeXGB = ids_pipeline_v18._SafeXGB

# Now load safely
bundle = joblib.load("./ids_model_v18.joblib")

print("Bundle version :", bundle["version"])
print("Internal classes:", list(bundle["label_encoder"].classes_))
print("Feature columns :", bundle["feature_cols"])
print("Top-K features  :", bundle["top_k_features"])
print("XGBoost LR      :", bundle["best_xgb_lr"])
print("XGBoost depth   :", bundle["best_xgb_depth"])
print("RF n_estimators :", bundle["best_rf_nest"])
print("RF max_depth    :", bundle["best_rf_depth"])
print("Internal metrics:", bundle["training_metrics"])

Bundle version : v18
Internal classes: ['Combined_Attack', 'ND_Attack', 'Normal', 'RA_Attack']
Feature columns : ['pkt_rate', 'na_rate', 'ra_burst_rate', 'ns_burst_rate', 'unique_src_count', 'unique_dst_count', 'mean_iat_ms', 'std_iat_ms', 'min_iat_ms', 'burstiness', 'multicast_ratio', 'src_diversity', 'ra_src_diversity', 'ns_unique_tgt_r', 'sliding_nd_ratio_3s']
Top-K features  : ['sliding_nd_ratio_3s', 'ns_burst_rate', 'pkt_rate', 'unique_src_count', 'unique_dst_count', 'src_diversity', 'multicast_ratio', 'ra_src_diversity', 'ra_burst_rate', 'burstiness']
XGBoost LR      : 0.3
XGBoost depth   : 4
RF n_estimators : 300
RF max_depth    : None
Internal metrics: {'XGBoost': {'bacc': 0.9034370364030584, 'mf1': 0.9094994240219789, 'roc': 0.9750817696512446, 'acc_te': 0.9418181818181818}, 'RF': {'bacc': 0.9085305403400662, 'mf1': 0.9135329538801004, 'roc': 0.9554813657232823, 'acc_te': 0.9454545454545454}}


In [6]:
import json

label_map = {
    "Normal": "Normal",
    "Attack": "RA_Attack",   # generic ICMPv6 attack mapped to closest class
}

label_map_json = json.dumps(label_map)
print(label_map_json)

{"Normal": "Normal", "Attack": "RA_Attack"}


In [7]:
!python ids_pipeline_v18.py --mode evaluate-csv --csv Labdataset.csv --label-column "Class" --label-map "{\"Normal\":\"Normal\",\"Attack\":\"RA_Attack\"}"

MODE: EVALUATE-CSV
  External CSV     : Labdataset.csv
  Model bundle in  : ./ids_model_v18.joblib
  Label column     : Class
  Loading external CSV: Labdataset.csv
  Raw CSV shape       : (1048575, 16)
  [WARN] External CSV missing 15/15 features.
         Filled with zeros: ['pkt_rate', 'na_rate', 'ra_burst_rate', 'ns_burst_rate', 'unique_src_count', 'unique_dst_count', 'mean_iat_ms', 'std_iat_ms', 'min_iat_ms', 'burstiness', 'multicast_ratio', 'src_diversity', 'ra_src_diversity', 'ns_unique_tgt_r', 'sliding_nd_ratio_3s']
         Model evaluated on DEGRADED feature set — report this in thesis.
  External label distribution:
    RA_Attack                : 1,045,624
    Normal                   : 2,951

──────────────────────────────────────────────────────────────────────
External evaluation — source: Labdataset.csv
──────────────────────────────────────────────────────────────────────

  Samples evaluated  : 1,048,575
  Present in external: ['Normal', 'RA_Attack']

  ── XGB ──
  Acc